# Ghostexec + Unsloth + GRPO (Colab)

This notebook mirrors the **install stack** from Unsloth’s [OpenEnv 2048 / GRPO Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game.ipynb): **`uv`**, **Unsloth + Unsloth Zoo from GitHub**, and pinned **`transformers==4.56.2`** + **`trl==0.22.2`** so `GRPOTrainer` imports cleanly (avoids newer TRL + `mergekit` + Pydantic issues on Colab).

**Hardware:** **GPU** runtime (T4 or better). **Runtime → Run all** from the top.

1. **Optional SFT** — synthetic briefing → `GhostexecAction` JSON (`smart_action` from `training/train.py`).
2. **GRPO** — reward from `training.grpo_ghostexec_reward.ghostexec_env_step_reward` (parse JSON → env `reset()` → one `step()`).

**Docs:** [Unsloth RL guide](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) · **Scope:** first-step comparisons on a deterministic reset (`phase2_core.json` by default).

**Repo:** set `GHOSTEXEC_REPO_URL` (or edit `GITHUB_REPO_URL` in the knobs cell) if you are not already inside the cloned `ghostexec` folder under `/content/ghostexec`.


## Installation (same pins as OpenEnv 2048 notebook)

Uses **`uv pip`** like the reference notebook: fresh Colab gets **torch**, **bitsandbytes**, **transformers 4.56.2**, **Unsloth from git**, then **`trl==0.22.2`** with **`--no-deps`** so pins stick.

Cells use **plain Python** (no `!` / `%%` magics) so the file stays easy to validate offline.


In [ ]:
import importlib.util
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], check=True)
try:
    import numpy as np

    get_numpy = f"numpy=={np.__version__}"
except Exception:
    get_numpy = "numpy"

colab = "COLAB_" in "".join(os.environ.keys())
no_torch = importlib.util.find_spec("torch") is None
if no_torch or colab:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "uv",
            "pip",
            "install",
            "-qqq",
            "torch>=2.8.0",
            "triton>=3.4.0",
            get_numpy,
            "torchvision",
            "bitsandbytes",
            "transformers==4.56.2",
            "trackio",
            "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo",
            "unsloth[base] @ git+https://github.com/unslothai/unsloth",
            "git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels",
        ],
        check=True,
    )
elif importlib.util.find_spec("unsloth") is None:
    subprocess.run(
        [sys.executable, "-m", "uv", "pip", "install", "-qqq", "unsloth", "trackio"],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "uv",
        "pip",
        "install",
        "--upgrade",
        "--no-deps",
        "transformers==4.56.2",
        "tokenizers",
        "trl==0.22.2",
        "unsloth",
        "unsloth_zoo",
    ],
    check=True,
)
print("Installed: Unsloth stack + transformers==4.56.2 + trl==0.22.2 (OpenEnv 2048 notebook pins).")


In [ ]:
import os

GITHUB_REPO_URL = os.environ.get("GHOSTEXEC_REPO_URL", "https://github.com/false200/ghostexec.git").strip()

RUN_SFT = os.environ.get("GHOSTEXEC_RUN_SFT", "1") != "0"
SFT_SAMPLES = int(os.environ.get("GHOSTEXEC_SFT_SAMPLES", "128"))
SFT_MAX_STEPS = int(os.environ.get("GHOSTEXEC_SFT_MAX_STEPS", "1"))
GRPO_DATASET_ROWS = int(os.environ.get("GHOSTEXEC_GRPO_ROWS", "24"))
GRPO_MAX_STEPS = int(os.environ.get("GHOSTEXEC_GRPO_MAX_STEPS", "2"))
NUM_GENERATIONS = int(os.environ.get("GHOSTEXEC_NUM_GENERATIONS", "4"))

MODEL_NAME = os.environ.get(
    "GHOSTEXEC_MODEL",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
)
MAX_SEQ_LENGTH = int(os.environ.get("GHOSTEXEC_MAX_SEQ", "2048"))


## Ghostexec repository

Clone into `/content/ghostexec` (Colab) or skip if you already uploaded the repo there. Then `pip install -e .` and **re-apply** the TRL / Transformers pins so dependencies from `pyproject.toml` do not float TRL upward.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

target = Path("/content/ghostexec")
candidates = [Path.cwd(), Path.cwd() / "ghostexec", target]
root = None
for p in candidates:
    if (p / "pyproject.toml").exists() and (p / "models.py").exists():
        root = p.resolve()
        break

if root is None:
    if not GITHUB_REPO_URL or "YOUR_ORG" in GITHUB_REPO_URL:
        raise RuntimeError(
            "Could not find ghostexec. Set GHOSTEXEC_REPO_URL / edit GITHUB_REPO_URL, or upload the repo."
        )
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(target)], check=True)
    root = target.resolve()

os.chdir(root)
sys.path.insert(0, str(root))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=str(root), check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "uv",
        "pip",
        "install",
        "--upgrade",
        "--no-deps",
        "transformers==4.56.2",
        "tokenizers",
        "trl==0.22.2",
        "unsloth",
        "unsloth_zoo",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install", "-qqq", "matplotlib", "datasets"],
    check=True,
)
print("Working directory:", root)


## Optional SFT: briefing → JSON action

Synthetic pairs from `smart_action` in `training/train.py` so the model learns valid `GhostexecAction` JSON before GRPO.


In [ ]:
import json
import random
from pathlib import Path

from ghostexec.server.ghostexec_environment import GhostexecEnvironment
from training.train import smart_action

ROOT = Path.cwd().resolve()
scenario = ROOT / "scenarios" / "phase2_core.json"
env = GhostexecEnvironment(scenario)

INSTRUCTION = (
    "You are an executive assistant agent. Reply with EXACTLY one JSON object (no markdown fences) "
    "with keys matching GhostexecAction: action_type (string), and optional email_id, message_body, "
    "meeting_id, new_time, reason, task_id, contact_name, message."
)


def build_sft_rows(n: int, seed: int = 0) -> list[dict]:
    rng = random.Random(seed)
    rows: list[dict] = []
    for _ in range(n):
        obs = env.reset()
        act = smart_action(obs, rng)
        user = INSTRUCTION + "\n\n=== BRIEFING ===\n" + (obs.echoed_message or "")
        assistant = json.dumps(act.model_dump(mode="json"), ensure_ascii=False)
        rows.append({"user": user, "assistant": assistant})
    return rows


sft_rows = build_sft_rows(min(SFT_SAMPLES, 512), seed=42)
sft_path = ROOT / "outputs" / "training" / "colab_sft_messages.jsonl"
sft_path.parent.mkdir(parents=True, exist_ok=True)
with sft_path.open("w", encoding="utf-8") as fh:
    for r in sft_rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print("Wrote", sft_path, "rows", len(sft_rows))


### SFT train (`SFTTrainer`)

Skipped when `RUN_SFT` is false. Uses 4-bit LoRA via Unsloth.


In [ ]:
import torch
from datasets import Dataset

model = None
tokenizer = None

if RUN_SFT:
    from unsloth import FastLanguageModel
    from trl import SFTConfig, SFTTrainer

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

    def to_text(example):
        msgs = [
            {"role": "user", "content": example["user"]},
            {"role": "assistant", "content": example["assistant"]},
        ]
        return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

    ds = Dataset.from_list(sft_rows)
    ds = ds.map(to_text, remove_columns=["user", "assistant"])

    _cuda = torch.cuda.is_available()
    _bf16 = _cuda and torch.cuda.is_bf16_supported()
    sft_args = SFTConfig(
        output_dir=str(ROOT / "outputs" / "training" / "unsloth_sft_ckpt"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        warmup_steps=1,
        max_steps=SFT_MAX_STEPS,
        logging_steps=1,
        learning_rate=2e-4,
        fp16=_cuda and not _bf16,
        bf16=_bf16,
        dataset_text_field="text",
        report_to="none",
    )
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=sft_args)
    trainer.train()
    print("SFT done.")
else:
    print("Skipping SFT (RUN_SFT=0).")


## GRPO with Ghostexec reward

`training.grpo_ghostexec_reward.ghostexec_env_step_reward` — fresh env, parse JSON, one `step()`. Dataset uses the same **chat-style** `prompt` list as the OpenEnv 2048 notebook (`[{role, content}]`).


In [ ]:
import torch
from datasets import Dataset
from pathlib import Path
from ghostexec.server.ghostexec_environment import GhostexecEnvironment
from training.grpo_ghostexec_reward import ghostexec_env_step_reward
from trl import GRPOConfig, GRPOTrainer

ROOT = Path.cwd().resolve()
scenario = ROOT / "scenarios" / "phase2_core.json"
_env = GhostexecEnvironment(scenario)
_brief = _env.reset().echoed_message or ""
GRPO_PROMPT = INSTRUCTION + "\n\n=== BRIEFING ===\n" + _brief

user_msg = {"role": "user", "content": GRPO_PROMPT.strip()}
grpo_ds = Dataset.from_list([{"prompt": [user_msg]}] * GRPO_DATASET_ROWS)

if model is None or tokenizer is None:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

maximum_length = len(tokenizer.apply_chat_template(grpo_ds[0]["prompt"], add_generation_prompt=True))
max_prompt_length = maximum_length + 4
max_completion_length = max(128, MAX_SEQ_LENGTH - max_prompt_length)

_cuda = torch.cuda.is_available()
_bf16 = _cuda and torch.cuda.is_bf16_supported()
training_args = GRPOConfig(
    temperature=1.0,
    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=GRPO_MAX_STEPS,
    save_steps=max(1, GRPO_MAX_STEPS),
    report_to="none",
    output_dir=str(ROOT / "outputs" / "training" / "unsloth_grpo_ckpt"),
    fp16=_cuda and not _bf16,
    bf16=_bf16,
    remove_unused_columns=False,
)

trainer_grpo = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[ghostexec_env_step_reward],
    args=training_args,
    train_dataset=grpo_ds,
)
trainer_grpo.train()
print("GRPO finished.")


In [ ]:
import matplotlib.pyplot as plt

try:
    log_hist = trainer_grpo.state.log_history
    keys = [k for h in log_hist for k in h if "reward" in k.lower() and "mean" in k]
    rewards = []
    for h in log_hist:
        for k in keys:
            if k in h:
                rewards.append(h[k])
                break
    if rewards:
        plt.figure(figsize=(8, 3))
        plt.plot(rewards, marker="o")
        plt.xlabel("log step")
        plt.ylabel("mean reward")
        plt.title("GRPO (Ghostexec)")
        plt.grid(True, alpha=0.3)
        plt.show()
    else:
        print("No reward mean keys in log_history; sample keys:", list(log_hist[0].keys()) if log_hist else [])
except Exception as e:
    print("Plot skip:", e)
